# CoworkOS 첫 실행: 메인 에이전트·코디네이터·서브에이전트로 나눈 오픈소스 cowork 하네스를 npm으로 설치하고 원하는 모델을 연결해 보기

위에서부터 차례로 실행해 보세요.

### Node 24 준비

In [ ]:
%%bash
npm i -g n && n 24 && hash -r && node -v

### CoWork OS 설치와 버전 확인

In [ ]:
%%bash
npm install -g cowork-os 2>&1 | grep -v deprecated | tail -5
cowork --version 2>&1 | tail -4

### Ollama 설치와 모델 받기

In [ ]:
%%bash
apt-get install -y zstd >/dev/null 2>&1
curl -fsSL https://ollama.com/install.sh | sh
(nohup ollama serve > ollama.log 2>&1 &); sleep 5
ollama pull qwen2.5:1.5b

### Ollama를 CoWork OS에 연결하고 한 번짜리 작업 실행

In [ ]:
%%bash
mkdir -p /work/ws && cd /work/ws
cowork providers configure ollama --base-url http://localhost:11434 --model qwen2.5:1.5b
cowork providers list 2>&1 | grep -v '^\[\|^  \|^}'
time cowork run "who are you?" --cwd /work/ws --provider ollama --model qwen2.5:1.5b --timeout-ms 240000 2>&1 | grep -v '^\[Db\|^\[Database\|^  \|^}' | tail -30

### 다단계 작업 실행 후 실행 기록 확인

In [ ]:
%%bash
cd /work/ws; printf 'alpha\n' > a.txt; printf 'beta notes\n' > b.md
cowork run "List the files in the current folder and write a summary file named summary.md describing each file." --cwd /work/ws --provider ollama --model qwen2.5:1.5b --permission-mode auto --timeout-ms 300000 > local_multistep.log 2>&1
grep -v '^\[Db\|^\[Database\|^  \|^}' local_multistep.log | tail -40
ID=$(cowork tasks list --cli 2>&1 | grep -v '^\[' | grep -i cancelled | head -1 | cut -d' ' -f1)
cowork tail $ID 2>&1 | grep -v '^\[Db\|^\[Database' | cut -c1-220 | head -50